# google_cloud_appengine_logging smoke test

Smoke test for `google-cloud-appengine-logging`, the protobuf message types
that App Engine request logs use in Cloud Logging (`RequestLog`, `LogLine`,
`SourceLocation`, `SourceReference`). Every example builds messages locally and
round-trips them; nothing is sent to Google Cloud.

## Example 1: Build a RequestLog

In [ ]:
import datetime

from google.cloud import appengine_logging

log = appengine_logging.RequestLog(
    app_id="demo",
    version_id="v1",
    method="GET",
    resource="/items?page=2",
    status=200,
    latency=datetime.timedelta(milliseconds=125),
    line=[appengine_logging.LogLine(log_message="hi")],
)
print(f"{log.method} {log.resource} -> {log.status} in {log.latency}")
assert log.app_id == "demo" and log.status == 200
assert log.latency == datetime.timedelta(milliseconds=125)  # a protobuf Duration, exposed as timedelta
assert len(log.line) == 1 and log.line[0].log_message == "hi"
assert log.request_id == "" and log.response_size == 0  # unset scalar fields read as their defaults
print("RequestLog built")

## Example 2: Wire-format round trip

In [ ]:
import datetime

from google.cloud import appengine_logging

log = appengine_logging.RequestLog(
    app_id="demo", method="POST", resource="/submit", status=201,
    latency=datetime.timedelta(milliseconds=40),
    line=[appengine_logging.LogLine(log_message="created")],
)
wire = appengine_logging.RequestLog.serialize(log)
print("serialized RequestLog:", len(wire), "bytes")
back = appengine_logging.RequestLog.deserialize(wire)
assert back == log
assert back.status == 201 and back.line[0].log_message == "created"
assert appengine_logging.RequestLog.pb(back).DESCRIPTOR.full_name == "google.appengine.logging.v1.RequestLog"
print("RequestLog round trip OK")

## Example 3: LogLine severity, timestamp and source location

In [ ]:
import datetime

from google.cloud import appengine_logging
from google.logging.type import log_severity_pb2

when = datetime.datetime(2026, 1, 2, 3, 4, 5, tzinfo=datetime.timezone.utc)
line = appengine_logging.LogLine(
    log_message="slow query",
    severity=log_severity_pb2.WARNING,
    time=when,
    source_location=appengine_logging.SourceLocation(file="app.py", line=42, function_name="handler"),
)
back = appengine_logging.LogLine.deserialize(appengine_logging.LogLine.serialize(line))
print(f"{back.time.isoformat()} severity={back.severity} {back.source_location.file}:{back.source_location.line}")
assert back.severity == log_severity_pb2.WARNING == 400
assert back.time == when
assert (back.source_location.file, back.source_location.line, back.source_location.function_name) == (
    "app.py", 42, "handler",
)
print("LogLine fields OK")

## Example 4: Source references

In [ ]:
from google.cloud import appengine_logging

log = appengine_logging.RequestLog(
    app_id="demo",
    source_reference=[
        appengine_logging.SourceReference(repository="https://example.com/repo.git", revision_id="abc123"),
    ],
)
back = appengine_logging.RequestLog.deserialize(appengine_logging.RequestLog.serialize(log))
ref = back.source_reference[0]
print("deployed from:", ref.repository, "@", ref.revision_id)
assert (ref.repository, ref.revision_id) == ("https://example.com/repo.git", "abc123")
assert len(back.source_reference) == 1
print("SourceReference OK")

## Example 5: JSON and dict forms

In [ ]:
import datetime
import json

from google.cloud import appengine_logging

log = appengine_logging.RequestLog(
    app_id="demo", method="GET", resource="/", status=404,
    latency=datetime.timedelta(milliseconds=5),
    line=[appengine_logging.LogLine(log_message="not found")],
)
as_json = appengine_logging.RequestLog.to_json(log)
parsed = json.loads(as_json)
# JSON uses lowerCamelCase field names and the string form of a Duration.
assert parsed["appId"] == "demo" and parsed["status"] == 404
assert parsed["latency"] == "0.005s"
assert parsed["line"][0]["logMessage"] == "not found"
assert appengine_logging.RequestLog.from_json(as_json) == log

as_dict = appengine_logging.RequestLog.to_dict(log)
assert as_dict["app_id"] == "demo" and as_dict["line"][0]["log_message"] == "not found"
print("JSON keys:", sorted(k for k, v in parsed.items() if v not in ("", "0", 0, [], False))[:8])
print("JSON and dict forms OK")